<a href="https://colab.research.google.com/github/sakuna47/RDB_DA/blob/DV_Code/RDB_DA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime
import seaborn as sns
from matplotlib.dates import MonthLocator, DateFormatter
import matplotlib.ticker as ticker

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Set style for better looking plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

def load_and_process_data(file_path):
    """
    Load and process the CSV file to extract all financial rates
    """
    try:
        # Try reading with different encodings and separators
        for encoding in ['utf-8', 'utf-8-sig', 'latin-1', 'cp1252']:
            try:
                df = pd.read_csv(file_path, encoding=encoding, header=None)
                print(f"Successfully loaded CSV with encoding: {encoding}")
                break
            except UnicodeDecodeError:
                continue
        else:
            df = pd.read_csv(file_path, header=None)
    except Exception as e:
        print(f"Error reading CSV file: {e}")
        for sep in [',', ';', '\t']:
            try:
                df = pd.read_csv(file_path, sep=sep, header=None)
                print(f"Successfully loaded CSV with separator: '{sep}'")
                break
            except:
                continue
        else:
            raise ValueError("Could not read the CSV file with any common format")

    print("Data shape:", df.shape)

     # Find key row indices
    date_row_idx = None
    usd_lkr_buying_idx = None
    usd_lkr_selling_idx = None
    treasury_bill_91_idx = None
    treasury_bill_182_idx = None
    treasury_bill_364_idx = None
    treasury_bond_2yr_idx = None
    treasury_bond_3yr_idx = None
    treasury_bond_4yr_idx = None
    treasury_bond_5yr_idx = None

 # Search for all required rows
    for idx, row in df.iterrows():
        row_str = ' '.join([str(cell) for cell in row if pd.notna(cell)])

        # Date row
        if 'Date' in str(row.iloc[0]) and '24.07.2024' in row_str:
            date_row_idx = idx
            print(f"Found date row at index: {idx}")

        # USD/LKR rates
        if 'USD/LKR Rate' in str(row.iloc[0]) and 'Buying' in str(row.iloc[1]):
            usd_lkr_buying_idx = idx
            print(f"Found USD/LKR Buying row at index: {idx}")
        elif usd_lkr_buying_idx is not None and idx == usd_lkr_buying_idx + 1 and 'Selling' in str(row.iloc[1]):
            usd_lkr_selling_idx = idx
            print(f"Found USD/LKR Selling row at index: {idx}")

        # Treasury Bills - Fixed pattern matching
        if 'Treasury Bills' in str(row.iloc[0]) and '91 Days' in str(row.iloc[1]):
            treasury_bill_91_idx = idx
            print(f"Found Treasury Bills 91 Days row at index: {idx}")
        elif str(row.iloc[1]).strip() == '182 Days' and treasury_bill_91_idx is not None:
            treasury_bill_182_idx = idx
            print(f"Found Treasury Bills 182 Days row at index: {idx}")
        elif str(row.iloc[1]).strip() == '364 Days' and treasury_bill_182_idx is not None:
            treasury_bill_364_idx = idx
            print(f"Found Treasury Bills 364 Days row at index: {idx}")

        # Treasury Bonds - Fixed pattern matching
        if 'Treasury Bonds' in str(row.iloc[0]) and '2 Yrs' in str(row.iloc[1]):
            treasury_bond_2yr_idx = idx
            print(f"Found Treasury Bonds 2 Yrs row at index: {idx}")
        elif str(row.iloc[1]).strip() == '3 Yrs' and treasury_bond_2yr_idx is not None:
            treasury_bond_3yr_idx = idx
            print(f"Found Treasury Bonds 3 Yrs row at index: {idx}")
        elif str(row.iloc[1]).strip() == '4 Yrs' and treasury_bond_3yr_idx is not None:
            treasury_bond_4yr_idx = idx
            print(f"Found Treasury Bonds 4 Yrs row at index: {idx}")
        elif str(row.iloc[1]).strip() == '5 Yrs' and treasury_bond_4yr_idx is not None:
            treasury_bond_5yr_idx = idx
            print(f"Found Treasury Bonds 5 Yrs row at index: {idx}")

    if date_row_idx is None:
        raise ValueError("Could not find date row in the data")